#### Late Arrival Test
- This is used to test that autoloader only loads newly arrived files
- After the initial setup where we loaded bronze data for day 1 and day 2, we will drop a late file in the transaction folder of the s3 day 2 folder. 
- It is expected that autolaoader only loads this new data when the pipeline is run not the already existing file.

In [0]:
%sql
--- Record a baseline i.e before the new load
SELECT COUNT(*) FROM dev_kyc.bronze.transactions;   -- 4,076
SELECT source_file, COUNT(*) FROM dev_kyc.bronze.transactions GROUP BY 1;

- Upload the late file transactions_late_002 in day 2 of S3 transaction folder and rerun the pipeline

#### Verify the new load

In [0]:
%sql
-- 4,076 + 150 = 4,226
SELECT COUNT(*) FROM dev_kyc.bronze.transactions;

In [0]:
%sql
-- The late file shows up as its own source, with a later load time
SELECT source_file, COUNT(*) AS row_count, MIN(ingest_ts) AS first_loaded
FROM dev_kyc.bronze.transactions
GROUP BY source_file
ORDER BY first_loaded;

In [0]:
%sql
-- How late the late rows are (txn_ts is a string in Bronze, so cast it safely)
SELECT COUNT(*) AS rows_late,
       MIN(try_cast(txn_ts AS TIMESTAMP)) AS earliest_txn,
       MAX(try_cast(txn_ts AS TIMESTAMP)) AS latest_txn,
       MIN(ingest_ts) AS loaded_at,
       ROUND(AVG((unix_timestamp(ingest_ts) - unix_timestamp(try_cast(txn_ts AS TIMESTAMP))) / 3600), 1) AS avg_hours_late
FROM dev_kyc.bronze.transactions
WHERE source_file LIKE '%late%';